In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline, AutoConfig
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    pipeline_device = torch.device('mps')
else:
    device = 'cpu'
    pipeline_device = -1

print({'selected_device': device})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='validation')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'split': 'validation', 'num_rows': len(dataset)})
print(dataset[:3])


In [ ]:
model_name = 'bhadresh-savani/distilbert-base-uncased-emotion'
config = AutoConfig.from_pretrained(model_name)

clf = pipeline(
    task='text-classification',
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True
)

id2label = {}
if hasattr(config, 'id2label') and config.id2label is not None:
    for k, v in config.id2label.items():
        id2label[int(k)] = str(v)

label2id = {}
if hasattr(config, 'label2id') and config.label2id is not None:
    label2id = {str(k): int(v) for k, v in config.label2id.items()}

print({'model_name': model_name, 'id2label': id2label})


In [ ]:
canonical_set = set(class_names)
alias_map = {
    'sadness': 'sadness',
    'sad': 'sadness',
    'joy': 'joy',
    'happy': 'joy',
    'love': 'love',
    'anger': 'anger',
    'angry': 'anger',
    'fear': 'fear',
    'scared': 'fear',
    'surprise': 'surprise',
    'surprised': 'surprise'
}

def normalize_label(raw_label):
    label = str(raw_label).strip()
    upper_label = label.upper()
    if upper_label.startswith('LABEL_'):
        idx = int(label.split('_')[-1])
        label = id2label.get(idx, label)
    label = str(label).strip().lower()
    label = alias_map.get(label, label)
    if label not in canonical_set:
        raise ValueError(f'Unrecognized label: {raw_label} -> {label}')
    return label

label_to_id = {name: i for i, name in enumerate(class_names)}
print({'class_names': class_names, 'label_to_id': label_to_id})


In [ ]:
texts = dataset['text']
true_ids = dataset['label']
batch_size = 32

pred_outputs = clf(texts, batch_size=batch_size, top_k=1)

pred_labels = []
pred_scores = []
for item in pred_outputs:
    if isinstance(item, list):
        item = item[0]
    pred_labels.append(normalize_label(item['label']))
    pred_scores.append(float(item['score']))

pred_ids = [label_to_id[label] for label in pred_labels]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': pred_labels,
    'score': pred_scores
})
results_df['correct'] = results_df['true_label'] == results_df['predicted_label']

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)
cm = confusion_matrix(true_ids, pred_ids)
cm_df = pd.DataFrame(cm, index=[f'true_{c}' for c in class_names], columns=[f'pred_{c}' for c in class_names])

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'validation',
    'num_examples': len(dataset),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report)
print(cm_df.to_string())


In [ ]:
thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
threshold_rows = []

for threshold in thresholds:
    accepted_mask = results_df['score'] >= threshold
    accepted_count = int(accepted_mask.sum())
    rejected_count = int((~accepted_mask).sum())
    coverage = accepted_count / len(results_df)
    if accepted_count > 0:
        accepted_accuracy = float(results_df.loc[accepted_mask, 'correct'].mean())
    else:
        accepted_accuracy = np.nan
    threshold_rows.append({
        'threshold': threshold,
        'accepted_examples': accepted_count,
        'rejected_examples': rejected_count,
        'coverage': round(float(coverage), 6),
        'accepted_accuracy': None if np.isnan(accepted_accuracy) else round(float(accepted_accuracy), 6)
    })

threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.to_string(index=False))


In [ ]:
uncertain_df = results_df[['text', 'true_label', 'predicted_label', 'score', 'correct']].copy()
uncertain_df = uncertain_df.sort_values(by='score', ascending=True).head(20).reset_index(drop=True)
uncertain_df['text'] = uncertain_df['text'].map(lambda x: x if len(x) <= 160 else x[:157] + '...')

print({'showing_most_uncertain_examples': len(uncertain_df)})
print(uncertain_df.to_string(index=False))


In [ ]:
focus_threshold = 0.90
accepted_df = results_df.loc[results_df['score'] >= focus_threshold, ['text', 'true_label', 'predicted_label', 'score', 'correct']].copy()
rejected_df = results_df.loc[results_df['score'] < focus_threshold, ['text', 'true_label', 'predicted_label', 'score', 'correct']].copy()

accepted_accuracy_focus = float(accepted_df['correct'].mean()) if len(accepted_df) > 0 else np.nan

print({
    'focus_threshold': focus_threshold,
    'accepted_examples': int(len(accepted_df)),
    'rejected_examples': int(len(rejected_df)),
    'accepted_accuracy': None if np.isnan(accepted_accuracy_focus) else round(float(accepted_accuracy_focus), 6)
})

accepted_preview = accepted_df.sort_values(by='score', ascending=False).head(10).reset_index(drop=True)
accepted_preview['text'] = accepted_preview['text'].map(lambda x: x if len(x) <= 140 else x[:137] + '...')
print(accepted_preview.to_string(index=False))

rejected_preview = rejected_df.sort_values(by='score', ascending=True).head(10).reset_index(drop=True)
rejected_preview['text'] = rejected_preview['text'].map(lambda x: x if len(x) <= 140 else x[:137] + '...')
print(rejected_preview.to_string(index=False))
